#  Animation数据可视化

### 一、数据理解
本数据集共收集了2024 年的热门动漫22000条数据

共7个字段说明

### 内容  

该数据集包含22个特征：  

- **Score（评分）**：分配给每部动漫的评级或分数。  
- **Popularity（流行度）**：衡量每部动漫在观众中的受欢迎程度。  
- **Rank（排名）**：数据集中每部动漫的排名。  
- **Members（会员数）**：与每部动漫相关的会员或观众数量。  
- **Description（简介）**：每部动漫的剧情和主题的简要概述或总结。  
- **Synonyms（别名）**：每部动漫使用的替代标题或别名。  
- **Japanese Title（日文标题）**：动漫的原日文标题。  
- **English Title（英文标题）**：动漫的英文翻译标题。  
- **Type（类型）**：动漫类型的分类（如电视剧、电影、OVA等）。  
- **Eps（集数）**：每部动漫系列的总集数。  
- **Status（状态）**：动漫的当前状态（如连载中、已完结等）。  
- **Aired（播出时间）**：动漫的播出日期范围。  
- **Premiered（首播时间）**：动漫首次首播的日期。  
- **Broadcast（播出平台）**：关于播出平台或频道的信息。  
- **Producers（制作方）**：参与制作动漫的公司或工作室。  
- **Licensors（版权方）**：持有动漫版权的组织或公司。  
- **Studios（制作公司）**：负责制作动漫的动画工作室。  
- **Source（原作）**：动漫的原始来源素材（如漫画、小说、原创等）。  
- **Genres（题材）**：动漫所属的类别或题材。  
- **Demographic（受众群体）**：动漫的目标受众（如少年、少女、青年、女性等）。  
- **Duration（时长）**：每集或每部电影的时长。  
- **Rating（分级）**：分配给每部动漫的内容分级（如G、PG、PG-13、R等）。

### 二、分析目的
1.动漫来源可视化 动漫来源占比（Pyecharts 饼图）
目的：明确各种动漫来源在数据集中的占比情况，了解主流动漫是基于什么制作的

2.热门动漫播出年份分析（Pyecharts 折线图）
目的：分析数据中动漫的播出年份，以确定每年的动漫数量和观众更喜欢那些年的动漫

3.热门动漫元素风格可视化（Pyecharts 词云图）
目的：分析数据中动漫的风格类型，以确定展示主流动漫风格元素

4.制作公司评分分布箱线图（Seaborn箱线图）
目的：对比不同制作公司动漫的评分分布，发现高分 / 低分类型的特征差异，分析哪些制作公司出品的动漫更受观众喜爱，质量更高

5.动漫年龄分级可视化（Pyecharts 柱状图）
目的：对比不同动漫的年龄分级，发现适合哪些年龄观众观看的动漫数量更多

6.不同动漫类型集数分析（Seaborn小提琴图）
目的：不同类型动漫集数分布分析

### 三、数据分析与可视化过程

#### 1、导入需要的库、编码、路径设置

In [ ]:
from pyecharts_exporter import display_chart

import os
import warnings as wn
from collections import Counter
from pyecharts import options as opts
from pyecharts.charts import WordCloud
import pandas as pd
import re  
import matplotlib.pyplot as plt
import numpy as np
import pyecharts.options as opts
import seaborn as sns
from pyecharts import options as opts
from pyecharts.charts import Bar
from pyecharts.charts import Funnel as fu
from pyecharts.charts import Line
from pyecharts.charts import Map as ma
from pyecharts.charts import Pie, WordCloud

plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

#### 2、导入数据并进行预处理



In [ ]:
df = pd.read_csv("data/Top_Anime_data.csv")


### 去除无关的数据列

In [ ]:
columns_to_drop = [
    "Description",
    "Synonyms",
    "Members",
    "Broadcast",
    "Producers",
    "Licensors",
    "Demographic",
    "English",
    "Duration",
    "Popularity",
]
df1 = df.drop(columns=columns_to_drop)
df1.head()

In [ ]:
df1.info()

### 重复值

In [ ]:
df1.duplicated().sum()

### 缺失值

In [ ]:
df1.isnull().sum()  # Premiered 有431个缺失值，属于正常现象，说明这些数据为电影或OVA无首播时间无需处理

#### 去掉属性列名的空格

In [ ]:
df1.columns

### 3.动漫来源可视化 动漫来源占比（Pyecharts 饼图）
目的：明确各种动漫来源在数据集中的占比情况，了解主流动漫是基于什么制作的

In [ ]:


# 统计每种 Source 的出现次数
source_counts = df1["Source"].value_counts().reset_index()
source_counts.columns = ["Source", "数量"]

# 取最后7种 Source，其他合并为"其他"类别
top_n = 7
if len(source_counts) > top_n:
    top_sources = source_counts.head(top_n)
    other_count = source_counts["数量"][top_n:].sum()
    top_sources = pd.concat(
        [top_sources, pd.DataFrame({"Source": ["其他"], "数量": [other_count]})]
    )
    source_counts = top_sources
data = [list(z) for z in zip(source_counts["Source"], source_counts["数量"])]
total = sum(source_counts["数量"])
percentages = [(item[0], round(item[1] / total * 100, 2)) for item in data]

# 创建饼图
pie = (
    Pie(init_opts=opts.InitOpts(width="1000px", height="600px"))
    .add(
        series_name="动漫来源",
        data_pair=data,
        radius=["40%", "75%"],  # 环形图内外半径
        center=["50%", "50%"],  # 图表中心位置
        label_opts=opts.LabelOpts(
            formatter="{b}: {c} ({d}%)",
            font_size=12,
        ),
        itemstyle_opts=opts.ItemStyleOpts(
            border_width=1, border_color="#fff"
        ),  # 分隔线
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="动漫来源类型分布",
            subtitle="来源(%占比)",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=20),
        ),
        legend_opts=opts.LegendOpts(
            type_="scroll",
            orient="vertical",
            pos_top="15%",
            pos_left="2%",
            textstyle_opts=opts.TextStyleOpts(font_size=12),
        ),
        toolbox_opts=opts.ToolboxOpts(
            is_show=True,
            feature={
                "saveAsImage": {},
                "dataView": {"readOnly": False},
                "restore": {},
                "magicType": {"type": ["pie", "funnel"]},
            },
        ),
        graphic_opts=opts.GraphicGroup(  # 增加文本标注
            graphic_item=opts.GraphicItem(left="8%", bottom="8%"),
            children=[
                opts.GraphicText(
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text="数据来源：Kaggle", font_size=14
                    )
                )
            ],
        ),
    )
)

display_chart(pie)



### 动漫来源类型分布饼图分析  
这张环形饼图统计了动漫作品的原始来源占比，核心结论如下：  

#### 1. 主导来源：Manga（漫画改编）  
- **占比 54.7%**，是绝对主流。说明漫画是动漫产业的 “内容基石”，大量动漫直接基于漫画 IP 开发，利用成熟的故事框架和粉丝基础降低创作风险。  
- 背后逻辑：漫画长期连载积累的剧情、角色和受众，能为动漫提供天然的内容储备，如《海贼王》《火影忍者》均是漫画改编的经典案例。  

#### 2. 第二梯队：Original（原创）+ Light novel（轻小说改编）  
- **Original（原创）占比 16%**：纯原创动漫虽占比不低，但开发难度高（需从 0 构建世界观、剧情），成功作品（如《EVA》《天元突破》）往往能成为现象级 IP，推动行业创新。  
- **Light novel（轻小说改编）占比 11%**：轻小说以文字为载体，擅长细腻叙事和人物塑造（如《刀剑神域》《魔法禁书目录》），适配动漫的 “青春、奇幻、校园” 等热门题材。  

#### 3. 小众来源：多元补充  
- **Novel（小说改编）6.3%**、**Web manga（网络漫画）3.2%** 等：反映行业对 “文字 IP” 和 “新兴漫画载体” 的尝试，但受众相对垂直。  
- **其他（4.4%）**：包含游戏改编、同人衍生等非主流来源，说明动漫产业边界在拓展，但尚未形成规模。  

 





### 4.热门动漫播出年份分析（Pyecharts 折线图）
目的：分析数据中动漫的播出年份，以确定每年的动漫数量和观众更喜欢那些年的动漫

In [ ]:
df1["Year"] = df1["Premiered"].str.extract(r"(\d{4})")
df1["Year"] = pd.to_numeric(df1["Year"], errors="coerce")
df2 = df1.dropna(subset=["Year"])
yearly_counts = df2["Year"].value_counts().sort_index().reset_index()
yearly_counts.columns = ["Year", "Count"]
x_data = yearly_counts["Year"].astype(str).tolist()
y_data = yearly_counts["Count"].tolist()


line = (
    Line()
    .add_xaxis(xaxis_data=x_data)
    .add_yaxis(
        series_name="每年动漫播出数量",
        y_axis=y_data,
        symbol="circle",
        is_symbol_show=True,
        label_opts=opts.LabelOpts(is_show=True),
    )
    .set_global_opts(
        tooltip_opts=opts.TooltipOpts(
            trigger="axis",
            axis_pointer_type="cross",  
            formatter="{b}年: {c}部"
        ),
        xaxis_opts=opts.AxisOpts(
            name="年份",
            type_="category",
            axislabel_opts=opts.LabelOpts(rotate=45, interval=0),
        ),
        yaxis_opts=opts.AxisOpts(
            name="数量",
            type_="value",
            axistick_opts=opts.AxisTickOpts(is_show=True),
        ),
        title_opts=opts.TitleOpts(
            title="动漫播出年份", subtitle="单位：部", pos_left="center"
        ),
        graphic_opts=opts.GraphicGroup(
            graphic_item=opts.GraphicItem(left="12%", bottom="0%"),
            children=[
                opts.GraphicText(
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text="数据来源：Kaggle", font="14px Microsoft YaHei"
                    )
                )
            ],
        ),
        legend_opts=opts.LegendOpts(pos_right="right", orient="vertical"),
        datazoom_opts=[  # 添加滑动条和框选缩放
            opts.DataZoomOpts(type_="slider"),
            opts.DataZoomOpts(type_="inside"),
        ],
    ))

display_chart(line)


### 动漫播出年份折线图分析  
折线图呈现了 1970 - 2024 年动漫播出数量的变化趋势，核心结论如下：  

#### 1. 早期（1970 - 2000 年）：缓慢起步，波动试探  
- **数量极低且不稳定**：70 - 90 年代每年播出量多为个位数，反映行业初期的 “小作坊式” 生产模式，内容以实验性、经典IP 试水为主（如《铁臂阿童木》《哆啦 A 梦》初代）。  
- **关键节点（2000 年后）**：2000 - 2010 年逐步爬坡，2005 年前后突破 10 部/年，与日本动漫产业商业化成熟（如 TV 动画档期常态化、周边开发完善）直接相关。  

#### 2. 增长期（2010 - 2020 年）：快速扩张，波动加剧  
- **2010 - 2020 年**：从 10 余部跃升至年播 30 部左右（2015、2018 年达 28 部），反映行业产能释放（技术进步、资本涌入）和市场需求爆发（流媒体平台扩张、全球化传播）。  
- **波动原因**：2012 年、2016 年等年份的数量回落，可能与 “热门 IP 空档期”“行业政策调整”（如内容审核、制作成本上涨）有关。  

#### 3. 近期（2020 - 2024 年）：峰值后回落，进入调整  
- **2023 年达峰值 39 部**，2024 年回落至 20 部：短期峰值可能是 “IP 集中开发”（如经典漫画续作、轻小说改编潮）的结果；回落则暗示行业进入 **“质量优先” 的调整期**（资方更谨慎，追求精品而非数量）。  
### 分析 从折线图中看出2024年热门动漫中,占比较多的是2005至2024年播出的动漫,2005年以前的动漫占比较少
### 可见观众们更喜欢近些年播出的动漫(新番)对与2005甚至是1997年以前播出的动漫(老番)喜爱程度较低

### 5.热门动漫元素风格可视化（Pyecharts 词云图）
目的：分析数据中动漫的风格类型，以确定展示主流动漫风格元素

In [ ]:
def split_genres(genre_str):
    if not isinstance(genre_str, str):
        return []
    
    # 处理常见的分隔符：逗号、分号、斜杠、空格（连续多个空格也算）
    genres = re.split(r'[,\s;/]+', genre_str)
    
    # 清洗标签：去除空白、去除重复前缀（如 ActionAction → Action）
    cleaned_genres = []
    for g in genres:
        g = g.strip()  # 去除首尾空格
        if not g:
            continue
            
        # 检测并修复重复标签（如 "ActionAction" → "Action"）
        match = re.match(r'^(\w+)\1$', g)
        if match:
            g = match.group(1)
            
        cleaned_genres.append(g)
    
    return cleaned_genres

# 2. 应用拆分函数并统计
all_genres = []
for genre_str in df['Genres'].dropna():
    genres = split_genres(genre_str)
    all_genres.extend(genres)

# 3. 统计词频
genre_counts = Counter(all_genres)
words = genre_counts.most_common(100)

# 创建词云图
wordcloud = (
    WordCloud(init_opts=opts.InitOpts(width="900px", height="600px"))
    .add(
        series_name="动漫风格类型",
        data_pair=words,
        word_size_range=[20, 100],
        shape="circle",
        rotate_step=45,
        textstyle_opts=opts.TextStyleOpts(font_family="SimHei"),  # 确保中文正常显示
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="动漫风格类型词云图",
            subtitle="基于独立风格标签的频率统计",
        ),
        graphic_opts=opts.GraphicGroup(
            graphic_item=opts.GraphicItem(left="12%", bottom="0%"),
            children=[
                opts.GraphicText(
                    graphic_textstyle_opts=opts.GraphicTextStyleOpts(
                        text="数据来源：Kaggle", font="14px Microsoft YaHei"
                    )
                )
            ],
        ),
    )
)


display_chart(wordcloud)


### 从这张动漫风格类型词云图中，我们可以分析出以下关键信息


### **1. 核心风格类型（高频词解读）**
- **`Action`（动作）**：字体最大，是**最主流风格**，说明动作元素在动漫中占绝对主导地位。  
- **`Drama`（剧情）**：第二大字体，剧情驱动的叙事风格极受欢迎。  
- **`Fantasy`（奇幻）**：高频出现，奇幻世界观（魔法、异世界）是热门题材。  如《葬送的芙莉莲》
- **`Comedy`（喜剧）**：排名靠前，轻松搞笑的内容需求旺盛。  


### **2. 潜力风格类型（中高频词）**
- **`Adventure`（冒险）**：结合 `Action`，“动作+冒险” 是经典组合（如《海贼王》）。  
- **`Sci-Fi`（科幻）**：科技/未来题材有稳定受众。  
- **`Romance`（恋爱）**：情感线是重要补充，常与其他风格融合（如 `Drama+Romance`）。  


### **3. 小众但独特的风格**
- **`Mystery`（悬疑）**、**`Supernatural`（超自然）**：虽字体较小，但说明悬疑/灵异题材有固定粉丝。  
- **`Slice of Life`（日常）**：生活流题材（如校园、职场）是重要分支，常作为 “治愈系” 存在。  




### 6.制作公司评分分布箱线图（Seaborn箱线图）
目的：对比不同制作公司动漫的评分分布，发现高分 / 低分类型的特征差异，分析哪些制作公司出品的动漫更受观众喜爱，质量更高


In [ ]:

# 筛选出评分和制作公司两列
df_score_studios = df1[['Score', 'Studios']].copy()

# 数据清洗：移除评分或制作公司为空的行
df_score_studios = df_score_studios.dropna(subset=['Score', 'Studios'])

# 统计每个制作公司的作品数量
studio_counts = df_score_studios['Studios'].value_counts()

# 筛选出作品数量较多的制作公司（例如，作品数 >= 10 的公司）
popular_studios = studio_counts[studio_counts >= 10].index
df_filtered = df_score_studios[df_score_studios['Studios'].isin(popular_studios)]

# 创建箱线图
plt.figure(figsize=(16, 8))
sns.boxplot(x='Studios', y='Score', data=df_filtered)
plt.title('不同制作公司的动漫评分分布')
plt.xlabel('制作公司')
plt.ylabel('评分')
plt.xticks(rotation=90)  # 旋转 x 轴标签避免重叠
plt.tight_layout()
plt.show()

从箱线图中，我们可以分析不同制作公司（Studios）的动漫评分（Score）分布特征，以下是具体解读：

### 1. 整体分布趋势
- **评分区间**：大部分制作公司的评分集中在 `8.0-8.8` 分区间，说明动漫行业整体评分水平较为稳定，但也存在一定差异。  
- **离散程度**：箱线图的“箱体”长度和“须线”（ whiskers ）长度差异明显，反映不同公司的评分稳定性不同。  


### 2. 头部制作公司分析（高评分、低离散）
- **White Fox**：  
  - 箱体偏上（中位数高），且箱体较短，**评分集中在 8.2-9.0 分**，说明该公司作品质量稳定且整体评分较高（如《Re：从零开始的异世界生活》《辉夜大小姐想让我告白》等代表作支撑）。  
  - 存在少量高分异常值（接近 9.4 分 ），可能是爆款作品拉高上限。  

- **Studio Ghibli（吉卜力工作室）**：  
  - 箱体位置较高，评分集中在 `8.2-8.6` 分，离散程度低（箱体短）。  
  - 作为经典动画工作室（如《千与千寻》《哈尔的移动城堡》），口碑稳定，评分波动小。

- **MAPPA**：  
  MAPPA 的箱线图整体位置偏上，**中位数接近 8.6**，说明其制作的动漫平均评分较高。  
  箱体（四分位距）较短，评分集中在 8.4 - 8.8 区间，**稳定性极佳**，几乎没有极端低分作品，体现出 MAPPA 对质量的把控能力。  
  （典型案例：《咒术回战》《进击的巨人 最终季》等高分作品，验证其 “高产优质” 特点 ）  

- **A-1 Pictures**：  
  箱线图显示 A-1 Pictures 的 **中位数约 8.2**，箱体跨度大（约 8.0 - 8.4 ），说明其作品评分差异较明显，存在 “高低分并存” 现象。  
  图中可见少量异常值（离散点），可能是口碑两极的作品（如粉丝向续作、实验性原创）。  

- **风格解读**：  
  A-1 Pictures 业务广泛（从《刀剑神域》系列到《辉夜大小姐想让我告白》到《青春猪头少年不会梦到兔女郎学姐》），既有 “流量保障” 的热门 IP 改编（评分稳定），也有尝试创新的作品（评分波动），体现其 “多元布局” 策略
### 3. （高上限、需稳定）
- **Madhouse**：  
  - 箱体较长（评分跨度大），但存在**极高异常值（接近 9.4 分 ）**，说明该公司既有高分神作（如《死亡笔记》《一拳超人》），也有评分偏低的作品，风格多元但稳定性不足。  

- **Bones**：  
  - 箱体偏上，且有多个高分点，代表作（如《钢之炼金术师》）拉高评分，但也有作品评分较低，需关注质量一致性。  


### 4. （稳定但缺乏亮点）
- **Kyoto Animation（京都动画）**：  
  - 评分集中在 `8.0-8.6` 分，箱体较短，**稳定性强**（如《轻音少女》《紫罗兰永恒花园》），但缺乏极致高分作品（无明显异常值），走“精品稳扎”路线。  

- **Toei Animation（东映动画）**：  
  - 箱体位置中等，评分离散小，作品数量多但风格偏大众（如《海贼王》长期连载），评分受受众基数大影响，整体平稳。  


### 5. （低评分、高离散）
- **SILVER LINK.**：  
  - 箱体偏下，且须线长（低分作品多），说明该公司作品整体评分较低，可能受题材、制作成本限制，需提升质量。  

- **P.A. Works**：  
  - 评分集中在 `8.0-8.4` 分，离散程度低但位置偏下，作品风格小众（如《花开伊吕波》），受众有限导致评分难突破。

- **J.C.STAFF**：  
  J.C.STAFF 的中位数约 8.0，箱体较长（7.8 - 8.2 区间为主），且存在较多离散点（异常值），说明其作品 **评分波动大、口碑分化严重**。

  J.C.STAFF 长期深耕校园、恋爱题材（如《灼眼的夏娜》《魔法禁书目录》），但近年也因 “产能过剩导致质量下滑” 被诟病（如某些季番的作画崩坏）。箱线图的离散点，正是 “经典 IP 高分 + 赶工作品低分” 的直观体现。  






### 7.动漫年龄分级可视化（Pyecharts 柱状图）
目的：对比不同动漫的年龄分级，发现适合哪些年龄观众观看的动漫数量更多

In [ ]:
# 统计不同年龄分级的动漫数量
rating_counts = df1['Rating'].value_counts().reset_index()
rating_counts.columns = ['Rating', 'Count']

# 创建柱状图
bar = (
    Bar(init_opts=opts.InitOpts(width="1000px", height="600px"))
    .add_xaxis(rating_counts['Rating'].tolist())
    .add_yaxis(
        "动漫数量",
        rating_counts['Count'].tolist(),
        label_opts=opts.LabelOpts(position="left"),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="动漫年龄分级分布",
            subtitle="展示不同年龄分级的动漫数量对比",pos_left='right'
        ),
         #添加脚注
        graphic_opts=opts.GraphicGroup(
            graphic_item=opts.GraphicItem(left='center',bottom='5%'),
            children=[
                opts.GraphicText(graphic_textstyle_opts=
                opts.GraphicTextStyleOpts(text='数据来源：Kaggle',
                                          font="14px Microsoft YaHei")
            )]
        ),
        xaxis_opts=opts.AxisOpts(
            name="年龄分级",
            axislabel_opts=opts.LabelOpts(rotate=45, font_size=10)
        ),
        yaxis_opts=opts.AxisOpts(
            name="数量",
            min_=0,
            axislabel_opts=opts.LabelOpts(formatter="{value}部")
        ,)
        
    )
)

# 渲染图表
display_chart(bar)


分析基于动漫年龄分级柱状图

### 一、核心数据分布（按分级排序）
| 年龄分级                | 数量（部） | 占比（%） | 典型含义                  |
|-------------------------|------------|-----------|---------------------------|
| **Teens 13 or older**   | 651        | 60.5      | 适合 13 岁及以上青少年    |
| **Violence & profanity** | 240        | 22.4      | 含暴力、脏话，建议成年观众|
| **G - All Ages**        | 54         | 5.0       | 全年龄段（儿童也可观看）  |
| **R+ - Mild Nudity**    | 45         | 4.2       | 含轻度裸露，18 岁以上适配 |
| **PG - Children**       | 10         | 0.9       | 儿童专属（家长指导级）    |  


### 二、深度分析：行业趋势与受众洞察
#### 1. 主流供给：青少年及成年向主导  
- **Teens 13 or older** 占比超 60%，是绝对主流。说明动漫产业核心瞄准 **青少年+年轻成人** 市场，这类作品题材更广（校园、奇幻、冒险），既能覆盖学生群体，也适配初入社会的年轻观众。  
- **Violence & profanity** 占 22.4%，反映 “成人向” 内容有稳定需求（如《进击的巨人》《咒术回战》），但因包含敏感元素，需分级限制。  


#### 2. 小众赛道：全年龄与儿童向供给不足  
- **G - All Ages** 仅 54 部，**PG - Children** 仅 10 部。儿童向动漫长期被《哆啦 A 梦》等经典 IP 垄断，新作品供给少；全年龄佳作（如《夏目友人帐》）稀缺，市场存在 **“低龄内容创新缺口”**。  


#### 3. 分级与题材的隐性关联  
- **Teens 13 or older** 多为 **成长、冒险、校园** 题材（如《鬼灭之刃》），用 “青春+战斗” 吸引青少年；  
- **Violence & profanity** 常涉及 **黑暗现实、社会讽刺**（如《心理测量者》），需年龄限制；  
- **G - All Ages** 以 **治愈、日常** 为主（如《治愈系食堂》），但产量极低，说明 “全年龄爆款” 难打造，制作方动力不足。  

### 8.不同动漫类型集数分析（Seaborn小提琴图）
目的：不同类型动漫集数分布分析

In [ ]:
import warnings
warnings.filterwarnings("ignore")


plt.style.use("ggplot")

# 创建小提琴图（优化颜色和显示）
ax = sns.violinplot(
    x="Type",
    y="Episodes",
    data=df,
    order=["TV", "Movie", "OVA", "TV Special", "ONA", "Special"],
    palette="Set2",  
    inner="quartile",  # 用四分位数显示内部，替代默认的均值线
    linewidth=1.5  # 增加轮廓线宽度，增强层次感
)

# 优化坐标轴和标题
plt.suptitle("不同类型动漫的集数分布", fontsize=14, fontweight="bold")
plt.title("单位：集", fontsize=10, y=1.02)  # 调整副标题位置
plt.xlabel("动漫类型", fontsize=12)
plt.ylabel("集数", fontsize=12)



# 显示图表
plt.tight_layout()  # 自动优化布局
plt.show()

### 不同类型动漫集数分布分析
从箱线图可清晰洞察各类动漫集数特征：  
- **TV 动漫**：集数跨度极大（28 - 1787 集），中位数低但离散点多，长剧集（如经典长篇番）拉高上限，反映 TV 类型既承载日常短番，也容纳超长篇叙事。  
- **Movie**：集数集中且极低（多为 1 集），符合电影 “单部完整叙事” 特性，少数离散点或为系列电影。  
- **OVA/ONA**：集数相对分散，OVA 中位数略高，二者均有中短剧集（如 1 - 50 集），适配补充剧情、网络独播等场景。  
- **TV Special/Special**：集数更紧凑，多为单集或短系列，用于特别篇、外传，特典等内容，满足粉丝额外需求。  

整体来看，不同类型动漫集数与发行场景强相关，TV 主导长篇生态，电影、特别篇聚焦短平快叙事，OVA/ONA 作为补充类型灵活适配多元需求 。 

### 四、总结

通过对动漫相关的饼图、折线图、箱线图、词云图、柱状图和小提琴图进行综合解读，可清晰洞察动漫行业的发展脉络与市场特征。  

从**来源分布饼图**看，超半数动漫改编自漫画（占比54.7% ），原创（16%）、轻小说改编（11%）等构成补充，漫画作为内容基石的地位显著，为动漫创作提供成熟IP与粉丝基础，不过也反映出行业对漫画依赖度高，原创内容突破空间大。  

**播出年份折线图**呈现出行业发展轨迹，早期产量低且波动，2010 - 2020年快速扩张，2023年达峰值后回落 。这既体现出技术进步、资本涌入带来的产能释放，也反映出行业从“数量扩张”向“质量竞争”的调整，2024年的回落或为长期健康发展蓄力。  

**制作公司箱线图**中，MAPPA等头部公司评分中位数高、波动小，质量把控佳；部分公司评分分化大，反映出不同制作方在产能与品质平衡上的差异，头部公司凭借稳定输出占据口碑优势，中小公司需在创新与质量间寻找突破。  

**风格词云图**里，动作、剧情、奇幻等风格突出，复合题材成主流 ，说明市场偏好多元融合，创作者需兼顾大众需求与风格创新，以贴合观众对丰富叙事的追求。  

**年龄分级柱状图**显示，青少年及成年向动漫占主导，儿童与全年龄向供给不足 。这与市场需求和商业回报博弈相关，也揭示出低龄、全龄赛道存在内容创新缺口，若能打造适配全家观看的作品，有望开辟新市场。  

**不同类型动漫集数分布小提琴图**展现出，TV 动漫集数跨度极大，长剧集与日常短番并存；Movie 集数集中且极低，契合单部完整叙事特点；OVA、ONA 等类型集数相对分散，适配补充剧情、网络独播等多元场景，反映出集数与发行场景的强关联性。  

综上，动漫行业以漫画为根基，风格多元但创新待突破，头部公司引领品质，产量进入调整期，不同类型动漫集数适配多样发行需求。未来需平衡质量与数量，挖掘小众赛道潜力，方能持续发展。

# 动漫行业可视化分析报告

## 一、选题背景
### （一）选题意义
结动漫产业在文化传播、文化消费等领域占据重要地位，结合本人个人爱好，对其进行数据分析与可视化，能洞察行业发展脉络，为创作者优化内容、制作方规划产能、平台精准运营提供依据，助力挖掘市场潜力、促进产业健康发展，也有助于大众理解动漫文化生态。

### （二）数据来源及伦理说明
数据源自Kaggle公开数据集，涵盖动漫的来源、播出年份、制作公司、风格、年龄分级、类型与集数等信息 。数据采集、使用遵循公开、合法原则，未涉及个人隐私等敏感内容，保证数据伦理合规。

## 二、数据分析流程说明
### （一）数据清洗
1. 处理缺失值：针对 `Score`（评分）、`Studios`（制作公司）、`Year`（年份）等关键字段，通过 `dropna` 方法移除含缺失值的行，确保分析数据完整。如对动漫播出年份分析时，过滤掉 `Year` 为空的数据。
2. 规范数据格式：提取动漫首播年份时，利用正则表达式 `df1["Year"] = df1["Premiered"].str.extract(r"(\d{4})")` 提取四位数字年份，再通过 `pd.to_numeric` 转换为数值类型，统一数据格式。

### （二）特征提取
从原始数据中筛选出与分析目标相关的特征，包括动漫来源类型（用于饼图分析）、播出年份（折线图）、制作公司与评分（箱线图）、风格标签（词云图）、年龄分级（柱状图）、类型与集数（小提琴图）等，构建多维度分析体系。

### （三）分析方法
1. 描述性统计分析：统计不同来源类型、年龄分级的动漫数量占比，计算制作公司评分的中位数、四分位数等，概括数据分布特征。
2. 可视化分析：运用饼图、折线图、箱线图、词云图、柱状图、小提琴图等多种图表，从不同维度直观呈现数据关系与趋势，辅助挖掘潜在规律。

## 三、可视化设计思路
### （一）图表类型选择依据
1. 饼图：适合展示动漫来源类型的占比关系，清晰呈现漫画、原创等不同来源的比例结构，突出漫画作为核心来源的地位。
2. 折线图：用于呈现动漫播出年份的数量变化趋势，便于观察行业发展的阶段性特征，如产量扩张与调整期。 
3. 箱线图：分析制作公司评分分布，通过中位数、四分位距等，对比头部公司与其他公司在质量把控上的差异。 
4. 词云图：展现动漫风格关键词的热度，直观反映市场偏好的风格类型，凸显复合题材的流行。 
5. 柱状图：对比不同年龄分级的动漫数量，清晰展示青少年向等主导类型与儿童、全年龄向的供给差异。 
6. 小提琴图：呈现不同类型动漫（TV、Movie等 ）的集数分布，体现集数与发行场景的关联及数据离散程度。 

### （二）配色方案
整体采用简洁、清新的配色风格，如饼图、柱状图等使用蓝色系为主色调，保证视觉舒适且突出数据。不同图表间色彩协调统一，同时通过颜色区分不同数据类别（如年龄分级柱状图中不同分级用不同深浅蓝色），增强可读性。

### （三）交互功能设计逻辑
部分图表（如基于Pyecharts的折线图）添加缩放、悬停提示、点击筛选等交互功能。缩放功能方便用户聚焦特定区间（如折线图中查看某几年的产量波动）；词云图中风格关键词的出现频次 ）；点击筛选用于突出查看特定类别（如柱状图中单独显示某一年龄分级的数量 ），

## 四、结论与启示
### （一）数据分析结果
1. 来源结构：超半数动漫改编自漫画（占比54.7%），原创、轻小说改编等为补充，漫画是内容基石，但原创突破空间大。 
2. 发展趋势：早期动漫产量低且波动，2010 - 2020年快速扩张，2023年达峰后回落，行业进入质量竞争调整期。 
3. 制作质量：MAPPA等头部公司评分稳定、质量佳，部分公司评分分化大，中小公司需平衡创新与质量。 
4. 风格偏好：动作、剧情、奇幻等风格突出，复合题材成主流，市场偏好多元融合叙事。 
5. 年龄分级：青少年及成年向动漫占主导，儿童与全年龄向供给不足，存在内容创新缺口。 
6. 集数分布：TV动漫集数跨度大，Movie集数集中且低，OVA、ONA等类型集数适配多元发行场景。 

### （二）对国家/社会问题的解决方案建议
1. 产业发展：鼓励原创内容创作，设立专项扶持基金或奖项，推动原创动漫IP孵化；引导制作方优化产能，平衡数量与质量，避免过度依赖单一来源（如漫画）。 
2. 市场拓展：挖掘儿童、全年龄向赛道潜力，支持制作适配全家观看的作品，结合传统文化元素，打造具有教育意义与娱乐性的内容，拓展受众群体。 

## 五、学习总结
### （一）课程知识应用体会
通过本次分析，将课程中学到的数据清洗、特征提取、可视化设计等知识，应用于实际动漫产业数据分析，深刻体会到理论知识在解决实际问题中的价值。不同图表类型的合理运用，能从多维度剖析数据，让行业特征与规律清晰呈现，提升了数据解读与决策辅助能力。

### （二）技术难点与解决思路
1. 数据清洗难点：原始数据中部分字段格式不统一（如首播年份提取 ）、存在缺失值，通过正则表达式精准提取信息，利用 `dropna` 等方法处理缺失值，规范数据格式。 
2. 可视化交互实现：在Pyecharts中配置交互功能时，需熟悉各类配置项（如 `tooltip_opts`、`toolbox_opts` ），通过查阅官方文档、调试代码，实现缩放、悬停提示等交互效果，增强图表实用性。 
